# Poisson solver

!!! note "Important points covered in this example"
      - Solving a volumetric problem
      - Using the method of manufactured solutions
      - ...

In [1]:
using Inti

## Problem definition

In this example we will solve the Poisson equation in a domain $\Omega$ with
Dirichlet boundary conditions on $\Gamma := \partial \Omega$:
$$
  \begin{align}
      \Delta u &= f  \quad \text{in } \Omega \\
      u &= g  \quad \text{on } \partial \Gamma
  \end{align}
$$
where $f : \Omega \to \mathbb{R}$ and $g : \Gamma \to \mathbb{R}$ are given
functions.

Seeking for a solution $u$ of the form ...

In [2]:
meshsize = 0.05
qorder = 5
interpolation_order = qorder

5

## Meshing

We now create the required meshes and quadratures for both $\Omega$ and $\Gamma$:

In [3]:
using Gmsh # this will trigger the loading of Inti's Gmsh extension

function gmsh_disk(; name, meshsize, order = 1, center = (0, 0), paxis = (2, 1))
    try
        gmsh.initialize()
        gmsh.option.setNumber("General.Terminal", 0)
        gmsh.model.add("circle-mesh")
        gmsh.option.setNumber("Mesh.MeshSizeMax", meshsize)
        gmsh.option.setNumber("Mesh.MeshSizeMin", meshsize)
        gmsh.model.occ.addDisk(center[1], center[2], 0, paxis[1], paxis[2])
        gmsh.model.occ.synchronize()
        gmsh.model.mesh.generate(2)
        gmsh.model.mesh.setOrder(order)
        gmsh.write(name)
    finally
        gmsh.finalize()
    end
end

name = joinpath(@__DIR__, "disk.msh")
gmsh_disk(; meshsize, order = 2, name)

Ω, msh = Inti.import_mesh_from_gmsh_file(name; dim = 2)
Γ = Inti.boundary(Ω)

Ωₕ = view(msh, Ω)
Γₕ = view(msh, Γ)

Info    : Reading '/home/lfaria/runner-integral-equations/_work/Inti.jl/Inti.jl/docs/src/examples/generated/disk.msh'...
Info    : 3 entities
Info    : 11983 nodes
Info    : 6089 elements
Info    : Done reading '/home/lfaria/runner-integral-equations/_work/Inti.jl/Inti.jl/docs/src/examples/generated/disk.msh'
┌ Warning: overwriting an existing entity with the same (dim,tag)=(0,1)
└ @ Inti ~/runner-integral-equations/_work/Inti.jl/Inti.jl/src/entities.jl:90
┌ Warning: overwriting an existing entity with the same (dim,tag)=(1,1)
└ @ Inti ~/runner-integral-equations/_work/Inti.jl/Inti.jl/src/entities.jl:90
┌ Warning: overwriting an existing entity with the same (dim,tag)=(2,1)
└ @ Inti ~/runner-integral-equations/_work/Inti.jl/Inti.jl/src/entities.jl:90


Inti.SubMesh{2, Float64} containing:
	 194 elements of type Inti.LagrangeElement{Inti.ReferenceHyperCube{1}, 3, StaticArraysCore.SVector{2, Float64}}

Use VDIM with the Vioreanu-Rokhlin quadrature rule
Q = Inti.VioreanuRokhlin(; domain = :triangle, order = qorder);
dict = Dict(E => Q for E in Inti.element_types(Ωₕ))

In [4]:
Ωₕ_quad = Inti.Quadrature(Ωₕ; qorder)
Γₕ_quad = Inti.Quadrature(Γₕ; qorder)

1164-element Inti.Quadrature{2, Float64}:
 Inti.QuadratureNode{2, Float64}([1.999999193998813, 0.0008515051397931395], 0.002965189101588837, [0.9999983929065215, 0.001792814651456313])
 Inti.QuadratureNode{2, Float64}([1.9999460276661267, 0.0073169829206595075], 0.009435136367433615, [0.9998925957649899, 0.014655952181637195])
 Inti.QuadratureNode{2, Float64}([1.9996572238731096, 0.01850570951186488], 0.012569886053405702, [0.9993169790955535, 0.036953691173374874])
 Inti.QuadratureNode{2, Float64}([1.9990134503593195, 0.031409865238500284], 0.012569893551151892, [0.9980329347374522, 0.06269179515174617])
 Inti.QuadratureNode{2, Float64}([1.9981872057178847, 0.04257179199460834], 0.00943515172670931, [0.9963864040030292, 0.08493605781946882])
 Inti.QuadratureNode{2, Float64}([1.9975965985367805, 0.049010469940377374], 0.0029651956882984732, [0.9952114135192774, 0.09774580502999537])
 Inti.QuadratureNode{2, Float64}([1.997427126804506, 0.05070499830106252], 0.0029650884972470645, [0.994

## Manufactured solution

For the purpose of comparing our numerical results to an exact solution, we
will use the method of manufactured solutions. For simplicity, we will take as
an exact solution

In [5]:
uₑ = (x) -> cos(2*x[1]) * sin(2*x[2])

#2 (generic function with 1 method)

which yields

In [6]:
fₑ = (x) -> -8 * uₑ(x)

#4 (generic function with 1 method)

Here is what the solution looks like:
qvals = map(Ωₕ_quad) do q
    return uₑ(q.coords)
end

ivals = Inti.quadrature_to_node_vals(Ωₕ_quad, qvals)

er = ivals - uₑ.(Ωₕ_quad.mesh.nodes)
norm(er,Inf)
Inti.write_gmsh_view(Ωₕ, uₑ.(Ωₕ.nodes))

## Boundary and integral operators

In [7]:
using FMMLIB2D

pde = Inti.Laplace(; dim = 2)

# Boundary operators
S_b2b, D_b2b = Inti.single_double_layer(;
    pde,
    target = Γₕ_quad,
    source = Γₕ_quad,
    compression = (method = :fmm, tol = 1e-12),
    correction = (method = :dim,),
)
S_b2d, D_b2d = Inti.single_double_layer(;
    pde,
    target = Ωₕ_quad,
    source = Γₕ_quad,
    compression = (method = :fmm, tol = 1e-12),
    correction = (method = :dim, maxdist = 5 * meshsize),
)

# Volume potentials
V_d2d = Inti.volume_potential(;
    pde,
    target = Ωₕ_quad,
    source = Ωₕ_quad,
    compression = (method = :fmm, tol = 1e-12),
    correction = (method = :dim, interpolation_order)
)
V_d2b = Inti.volume_potential(;
    pde,
    target = Γₕ_quad,
    source = Ωₕ_quad,
    compression = (method = :fmm, tol = 1e-12),
    correction = (method = :dim, maxdist = 5 * meshsize, interpolation_order),
)

[ Info: Loading Inti.jl FMMLIB2D extension


1164×41258 LinearMaps.LinearCombination{Float64} with 2 maps:
  1164×41258 LinearMaps.FunctionMap{Float64,true}(#3; issymmetric=false, ishermitian=false, isposdef=false)
  1164×41258 LinearMaps.WrappedMap{Float64} of
    1164×41258 SparseArrays.SparseMatrixCSC{Float64, Int64} with 8148 stored entries

We can now solve a BIE for the unknown density $\sigma$:

In [8]:
f = map(Ωₕ_quad) do q
    return fₑ(q.coords)
end
g = map(Γₕ_quad) do q
    return uₑ(q.coords)
end
rhs = V_d2b * f + g

using LinearAlgebra
L = -I / 2 + D_b2b

1164×1164 LinearMaps.LinearCombination{Float64} with 3 maps:
  1164×1164 LinearMaps.UniformScalingMap{Float64} with scaling factor: -0.5
  1164×1164 LinearMaps.FunctionMap{Float64,true}(#4; issymmetric=false, ishermitian=false, isposdef=false)
  1164×1164 LinearMaps.WrappedMap{Float64} of
    1164×1164 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6984 stored entries

If `compression=none` or `compresion=hmatrix` is used above for constructing `D_b2b`, we could alternately use dense linear algebra:

In [9]:
#F = lu(L)
#σ = F \ rhs

using IterativeSolvers
σ, hist =
    gmres(L, rhs; log = true, abstol = 1e-10, verbose = false, restart = 100, maxiter = 100)
@show hist

hist = Converged after 4 iterations.


Converged after 4 iterations.

To check the solution, lets evaluate it at the nodes $\Omega$

In [10]:
uₕ_quad = -(V_d2d * f) + D_b2d * σ
uₑ_quad = map(q -> uₑ(q.coords), Ωₕ_quad)
er = abs.(uₕ_quad - uₑ_quad)
@show norm(er, Inf)

norm(er, Inf) = 4.291893582450951e-8


4.291893582450951e-8

## Visualize the solution error using Gmsh
er_nodes = Inti.quadrature_to_node_vals(Ωₕ_quad, er)

In [11]:
sol_nodes = uₑ.(Inti.nodes(Ωₕ))
solₕ_nodes = Inti.quadrature_to_node_vals(Ωₕ_quad, uₑ_quad)
er_nodes = abs.(sol_nodes - solₕ_nodes)

gmsh.initialize()
Inti.write_gmsh_model(msh)
Inti.write_gmsh_view!(Ωₕ, er_nodes; name="error")

[ Info: Inti.LagrangeElement{Inti.ReferenceSimplex{2}, 6, StaticArraysCore.SVector{2, Float64}}


Inti.write_gmsh_view!(Ωₕ, sol_nodes; name="solution")

In [12]:
"-nopopup" in ARGS || gmsh.fltk.run()
gmsh.finalize()

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*